In [1]:
#kernel thesis clean
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_pickle("screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [11]:
x_data = np.array(df['torque_values'].tolist())
y_data = np.array(df['class_values'].tolist())

le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.25,stratify=y_train_full,random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1)
X_test  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: torch.Size([7500, 800, 1]) torch.Size([7500])
Validation: torch.Size([2500, 800, 1]) torch.Size([2500])
Test: torch.Size([2500, 800, 1]) torch.Size([2500])


In [12]:
x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape)
print(y_batch.shape)

torch.Size([32, 800, 1])
torch.Size([32])


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class TransformerModel(nn.Module):
    def __init__(self, input_dim: int, d_model: int = 64, nhead: int = 8, num_layers: int = 2, dropout: float = 0.1, output_dim: int = 8):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=4 * d_model, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, output_dim) #gibt die 8 klassen zurück

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x) 
        class_out = x.mean(dim=1) # hat die form (batch, d_model) und gibt für jeden Zeitschritt die Klassenvorhersage aus
        class_out = self.head(class_out) #hat die form (batch, output_dim) 
        return class_out

In [4]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
from sklearn.metrics import f1_score
import math


model = TransformerModel(input_dim=1, d_model=64, nhead=8, num_layers=2, dropout=0.1, output_dim=8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

earlystop = EarlyStopper(patience=5, min_delta=0.001)

epochs = 50

train_losses = []
val_losses = []
val_f1_scores = []

best_overall_f1 = -np.inf
best_overall_model_state = None

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()
        
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_val_batch, y_val_batch in val_loader:
            X_val_batch = X_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            outputs = model(X_val_batch)
            loss = criterion(outputs, y_val_batch)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_val_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1_scores.append(val_f1)
    scheduler.step(avg_val_loss)

    #best model speichern für test
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()

    if earlystop.early_stop(avg_val_loss):
        print("Early stopping triggered")
        break

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1 Score: {val_f1:.4f}")


Epoch 1/50, Train Loss: 1.9758, Val Loss: 1.9333, Val F1 Score: 0.1273
Epoch 2/50, Train Loss: 1.8121, Val Loss: 1.7365, Val F1 Score: 0.2354
Epoch 3/50, Train Loss: 1.7121, Val Loss: 1.6865, Val F1 Score: 0.2344
Epoch 4/50, Train Loss: 1.6884, Val Loss: 1.6674, Val F1 Score: 0.2527
Epoch 5/50, Train Loss: 1.6653, Val Loss: 1.6479, Val F1 Score: 0.2470
Epoch 6/50, Train Loss: 1.6577, Val Loss: 1.6500, Val F1 Score: 0.2452
Epoch 7/50, Train Loss: 1.6470, Val Loss: 1.6746, Val F1 Score: 0.2306
Epoch 8/50, Train Loss: 1.6309, Val Loss: 1.6348, Val F1 Score: 0.2984
Epoch 9/50, Train Loss: 1.6249, Val Loss: 1.6249, Val F1 Score: 0.2611
Epoch 10/50, Train Loss: 1.6153, Val Loss: 1.6464, Val F1 Score: 0.2723
Epoch 11/50, Train Loss: 1.6070, Val Loss: 1.6352, Val F1 Score: 0.2839
Epoch 12/50, Train Loss: 1.6104, Val Loss: 1.6176, Val F1 Score: 0.2770
Epoch 13/50, Train Loss: 1.6023, Val Loss: 1.5903, Val F1 Score: 0.2895
Epoch 14/50, Train Loss: 1.5823, Val Loss: 1.5974, Val F1 Score: 0.2726
E

In [15]:
model.load_state_dict(best_model_state)
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 1.5041
Test F1 Macro: 0.3411


## Cross validation

In [6]:
from sklearn.metrics import f1_score
import math

In [7]:
x_data = np.array(df['torque_values'].tolist())
y_data = np.array(df['class_values'].tolist())

le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_scores = []

best_overall_model_state = None
best_overall_f1 = -np.inf


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"Fold {fold+1}")

    X_train_fold = X_train_full[train_idx]
    y_train_fold = y_train_full[train_idx]

    X_val_fold = X_train_full[val_idx]
    y_val_fold = y_train_full[val_idx]

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32).unsqueeze(-1),torch.tensor(y_train_fold, dtype=torch.long)),batch_size=32,shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32).unsqueeze(-1),torch.tensor(y_val_fold, dtype=torch.long)),batch_size=32,shuffle=False)

    model = TransformerModel(input_dim=1,d_model=64,nhead=8,num_layers=2,dropout=0.1,output_dim=8).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    earlystop = EarlyStopper(patience=7, min_delta=0.001)

    epochs = 50

    best_val_f1 = -np.inf
    best_model_state = None


    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                X_val_batch = X_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                outputs = model(X_val_batch)
                loss = criterion(outputs, y_val_batch)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_val_batch.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        scheduler.step(avg_val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()

        if earlystop.early_stop(avg_val_loss):
            print("Early stopping triggered")
            break

        print(f"Fold {fold+1}, Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")

    cv_scores.append(best_val_f1)
    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_model_state = best_model_state


print(f"CV f1 mean avg score: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")

Fold 1
Fold 1, Epoch 1/50, Train Loss: 1.9811, Val Loss: 1.9204, Val F1: 0.1455
Fold 1, Epoch 2/50, Train Loss: 1.8167, Val Loss: 1.7579, Val F1: 0.2287
Fold 1, Epoch 3/50, Train Loss: 1.7018, Val Loss: 1.6926, Val F1: 0.2169
Fold 1, Epoch 4/50, Train Loss: 1.6820, Val Loss: 1.7332, Val F1: 0.2068
Fold 1, Epoch 5/50, Train Loss: 1.6665, Val Loss: 1.6681, Val F1: 0.2325
Fold 1, Epoch 6/50, Train Loss: 1.6567, Val Loss: 1.6787, Val F1: 0.2475
Fold 1, Epoch 7/50, Train Loss: 1.6463, Val Loss: 1.6708, Val F1: 0.2406
Fold 1, Epoch 8/50, Train Loss: 1.6391, Val Loss: 1.6522, Val F1: 0.2557
Fold 1, Epoch 9/50, Train Loss: 1.6347, Val Loss: 1.6544, Val F1: 0.2525
Fold 1, Epoch 10/50, Train Loss: 1.6201, Val Loss: 1.6621, Val F1: 0.2849
Fold 1, Epoch 11/50, Train Loss: 1.6185, Val Loss: 1.6496, Val F1: 0.2678
Fold 1, Epoch 12/50, Train Loss: 1.6066, Val Loss: 1.6775, Val F1: 0.2523
Fold 1, Epoch 13/50, Train Loss: 1.6058, Val Loss: 1.7012, Val F1: 0.2673
Fold 1, Epoch 14/50, Train Loss: 1.6073,

In [8]:
final_model = TransformerModel(input_dim=1, d_model=64, nhead=8, num_layers=2, dropout=0.1, output_dim=8)
final_model.to(device)
final_model.load_state_dict(best_overall_model_state)
final_model.eval()

test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1),torch.tensor(y_test, dtype=torch.long)),batch_size=32,shuffle=False)

all_preds = []
all_labels = []
test_loss = 0.0

criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:

        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)
        outputs = final_model(X_test_batch)
        loss = criterion(outputs, y_test_batch)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"Final Test F1 Macro: {test_f1:.4f}")

Final Test F1 Macro: 0.3496
